In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
from tabulate import tabulate
from astropy.time import Time
from tqdm import tqdm
import csv
import os
import glob
import multiprocessing

In [ ]:
import glob
import pandas as pd

# Initial hit count (absolute path)
print('initial hit count')
for file in glob.glob('/datax/scratch/ellambishop/test_refine/*.pkl'):
    df = pd.read_pickle(file)
    print(f"{file}: {len(df)} rows")

# Cleaned hit count (relative path from current directory)
print('\ncleaned hit count')
for file in glob.glob('clean_data/*.pkl'):
    df = pd.read_pickle(file)
    print(f"{file}: {len(df)} rows")



In [ ]:
import matplotlib.pyplot as plt

# Categories
categories = ["VLASS Coherent", "Non-VLASS Coherent"]

# Initial counts
initial = [1426+420344, 845960+1942776, ]  # total hits

# Cleaned counts (clean + unmatched clean)
clean = [110+36261, 592+33869]  # sum of clean + clean unmatched

# Maybe RFI counts (maybe_rfi + unmatched maybe)
maybe_rfi = [40092+252700, 667374+157555]

# Other (remaining hits not accounted in clean or maybe)
other = [initial[0] - (clean[0] + maybe_rfi[0]),
         initial[1] - (clean[1] + maybe_rfi[1])]

# Plot stacked bar
plt.figure(figsize=(10,6))
plt.bar(categories, clean, label='Clean', alpha=0.7)
plt.bar(categories, maybe_rfi, bottom=clean, label='Maybe RFI', alpha=0.7)
plt.bar(categories, other, bottom=[c+m for c,m in zip(clean, maybe_rfi)],
        label='Other', alpha=0.4)
plt.ylabel("Number of Hits")
plt.title("VLASS and Non-VLASS Coherent Hit Reduction")
plt.legend()
plt.show()


In [ ]:
# 3️⃣ Unmatched
categories = ["VLASS Unmatched", "Non-VLASS Unmatched"]

# Initial counts for unmatched only
initial = [36261 + 252700, 33869 + 157555]

# Cleaned counts (only unmatched clean)
clean = [36261, 33869]

# Maybe RFI counts (only unmatched maybe_rfi)
maybe_rfi = [252700, 157555]

# Other (remaining hits not accounted in clean or maybe)
other = [initial[0] - (clean[0] + maybe_rfi[0]),
         initial[1] - (clean[1] + maybe_rfi[1])]

plt.figure(figsize=(8,5))
plt.bar(categories, clean, label='Clean', alpha=0.7)
plt.bar(categories, maybe_rfi, bottom=clean, label='Maybe RFI', alpha=0.7)
plt.bar(categories, other, bottom=[c+m for c,m in zip(clean, maybe_rfi)],
        label='Other', alpha=0.4)
plt.ylabel("Number of Hits")
plt.title("Unmatched VLASS and Non-VLASS Hit Reduction")
plt.ylim(0,350000)
plt.legend()
plt.savefig('unmatched_coh.jpg')

In [3]:
clean_df = pd.read_pickle('/datax/scratch/ellambishop/data/clean_non_vlass.pkl')

In [4]:
# signal density/ timevs freq
rfi_only = clean_df[clean_df['rfi_flag']==True]
plt.figure(figsize=(12, 6))
plt.hexbin(rfi_only['tstart'], rfi_only['signal_frequency'], gridsize=200, cmap='Reds')
plt.xlabel("Time Start")
plt.ylabel("Frequency (MHz)")
plt.title("RFI Flagged Signal Density: Time vs Frequency")
plt.colorbar(label='Hit Count')
#plt.show()
plt.close()

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Load signal data (for histogram)
df = clean_df

# Load interval data (to shade)
intervals = pd.read_csv("RFI_band_files/Known_s-band_RFI.csv")
print("CSV columns:", intervals.columns.tolist())
# Start the plot
plt.figure(figsize=(12, 6))


# Plot histogram
sns.histplot(
    data=df,
    x="signal_frequency",
    #hue="date",
    element="step",
    stat="density",
    common_norm=False
)

# Add shaded regions and text labels
for _, row in intervals.iterrows():
    # Draw the shaded band
    plt.axvspan(row['start_frequency'], row['end_frequency'], color='gray', alpha=0.3)

    # Add the label at the center of the span
    center_x = (row['start_frequency'] + row['end_frequency']) / 2
    plt.text(
        center_x,
        plt.ylim()[1] * 0.95,  # near top of plot
        row['label'],
        ha='center',
        va='top',
        fontsize=10,
        color='black',
        alpha=0.8,
        rotation=90  # optional: vertical text
    )

# Add legend patch manually for shaded regions
#shaded_patch = Patch(facecolor='gray', alpha=0.3, label='RFI')
#plt.legend(handles=plt.gca().get_legend_handles_labels()[0] + [shaded_patch])

# Final touches
plt.xlabel("Signal Frequency (MHz)")
plt.ylabel("Density")
#plt.xlim(5800,7500)
plt.title("Signal Frequency Histogram with Shaded & Labeled Intervals")
plt.tight_layout()
#plt.show()
plt.close()

CSV columns: ['start_frequency', 'end_frequency', 'label']


In [7]:
#visualize signal vs power with snr value incuded 
def pow_freq_snr(full_df):
    df_new = full_df
    sns.set_theme()
    sns.relplot(data=df_new, x=df_new["signal_frequency"], y=df_new["signal_power"], 
                hue=df_new["signal_snr"], size = df_new["signal_snr"], palette = "ch:s=.25,rot=-.25")
 
    plt.xlabel("Signal Frequency MHz")
    plt.ylabel("Signal Power")
    plt.title("Signal vs Power and SNR")
    plt.grid(True)
    plt.show()

#pow_freq_snr(clean_df)

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis, skew

# Drop NaNs from power_ratio
ratios = clean_df['power_ratio'].dropna()

# Calculate stats
excess_kurt = kurtosis(ratios, fisher=True)
skewness = skew(ratios)

print("Excess Kurtosis:", excess_kurt)  # >0 = heavy tail
print("Skewness:", skewness)

# Plot histogram
plt.figure(figsize=(10,6))
sns.histplot(data=clean_df, x='power_ratio', hue='rfi_flag', bins=50, element='step')

# Annotate plot
#plt.text(0.7, plt.ylim()[1]*0.8, f'Excess Kurtosis: {excess_kurt:.2f}\nSkewness: {skewness:.2f}', fontsize=10)
plt.title("Distribution of Power Ratios")
plt.xlabel("Coherent / Incoherent Power")
plt.ylabel("Count")
plt.yscale('log')
plt.grid(True)
#plt.show()
plt.close()

Excess Kurtosis: -0.6977405011017148
Skewness: 0.01609562021612501
